# 특징 중요도 실습

**Feature Importance · Permutation Importance · 순열 중요도**

어떤 입력 변수가 모델 성능이나 예측에 큰 영향을 주는지 나타내는 값. 인과관계를 뜻하지는 않는다.

소재 분야에서 이해하기: 변수를 섞었을 때 오차가 가장 커지는 기술자를 확인한다.

이 노트북은 개념을 직접 돌려보기 위한 예제입니다. 데이터는 실제 측정값이 아니라 개념 확인용으로
생성한 값이므로, 결과 수치를 연구 결론으로 쓰지 마세요. 위에서부터 순서대로 실행하세요.
그림의 축 이름은 기본 폰트에 한글 글리프가 없어 영문으로 적었습니다.

참고 자료: [scikit-learn 순열 중요도 문서](https://scikit-learn.org/stable/modules/permutation_importance.html)

## 1. 순열 중요도

변수를 무작위로 섞었을 때 성능이 얼마나 나빠지는지로 중요도를 봅니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (7, 4)

def make_alloy_data(n=240, noise=6.0, seed=0):
    """개념 확인용 합성 데이터. 실제 합금 측정값이 아닙니다.

    x1 소성 온도(600-900 C), x2 유지 시간(0.5-8 h), x3 첨가 원소 비율(0-5 at%),
    x4 측정 노이즈만 담긴 무의미한 변수. y 는 경도(HV) 를 흉내낸 값입니다.
    """
    rng = np.random.default_rng(seed)
    x1 = rng.uniform(600, 900, n)
    x2 = rng.uniform(0.5, 8.0, n)
    x3 = rng.uniform(0.0, 5.0, n)
    x4 = rng.normal(0.0, 1.0, n)
    y = (120 + 0.14 * (x1 - 600) + 9.0 * np.sqrt(x2) + 11.0 * x3
         - 0.9 * x3 ** 2 - 0.004 * (x1 - 750) * x2 + rng.normal(0, noise, n))
    X = np.column_stack([x1, x2, x3, x4])
    return X, y, ['소성온도', '유지시간', '첨가비율', '무관변수']


X, y, FEATURES = make_alloy_data()
print(X.shape, y.shape, FEATURES)
print('경도 평균 %.1f, 표준편차 %.1f' % (y.mean(), y.std()))

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)
model = RandomForestRegressor(n_estimators=300, random_state=0).fit(X_train, y_train)

result = permutation_importance(model, X_test, y_test, n_repeats=20, random_state=0)
order = np.argsort(result.importances_mean)
plt.barh(range(len(FEATURES)), result.importances_mean[order], xerr=result.importances_std[order])
plt.yticks(range(len(FEATURES)), [['temp', 'time', 'additive', 'noise'][i] for i in order])
plt.xlabel('drop in R2 when shuffled'); plt.show()
for index in order[::-1]:
    print('%-8s 중요도 %.3f ± %.3f' % (FEATURES[index], result.importances_mean[index], result.importances_std[index]))

## 2. 상관된 변수는 중요도를 나눠 갖습니다

In [ ]:
X_dup = np.column_stack([X, X[:, 0] + rng.normal(0, 1, len(X))])   # 온도와 거의 같은 변수 추가
names = FEATURES + ['온도복제']
X_tr, X_te, y_tr, y_te = train_test_split(X_dup, y, test_size=0.3, random_state=0)
model2 = RandomForestRegressor(n_estimators=300, random_state=0).fit(X_tr, y_tr)
result2 = permutation_importance(model2, X_te, y_te, n_repeats=20, random_state=0)
for name, value in zip(names, result2.importances_mean):
    print('%-8s 중요도 %.3f' % (name, value))
print('\n온도의 중요도가 복제 변수와 나뉘어 낮아 보입니다. 중요도가 낮다고 무관하다는 뜻이 아닙니다.')
print('또한 중요도는 예측에 대한 기여이며 인과관계가 아닙니다.')

---

셀의 숫자를 바꿔가며 다시 실행해보면 개념이 더 분명해집니다. 용어 사전으로 돌아가려면
[소재·AI 용어 사전](https://forum.rnddata.org/glossary/#feature-importance)을 여세요.